# Feature Engineering for Phishing URL Detection

## 1. Introduction

Feature engineering is the process of transforming existing data into meaningful features that can provide additional information to machine learning models.

In this phishing URL detection project, the PhiUSIIL dataset already contains several URL, domain, and webpage-related characteristics. Additional derived features can be created from these existing variables to represent useful relationships such as ratios, densities, and structural properties.

## 2. Objectives

The main objectives of this notebook are:

- Examine the existing predictive features.
- Identify meaningful opportunities for feature engineering.
- Create additional URL and webpage-related features.
- Preserve the original predictive features.
- Avoid unnecessary duplication of information.
- Prevent target-variable leakage.
- Check the quality of the newly created features.
- Prepare the engineered feature set for feature selection and model development.

## 3. Dataset

The project uses the PhiUSIIL Phishing URL Dataset.

The dataset contains:

- 235,795 records
- 56 total columns
- 50 numerical predictive features
- 5 text/categorical columns
- 1 target variable: `label`
- `0` = Phishing
- `1` = Legitimate

The `FILENAME` column is metadata and will not be used as a predictive feature.

## 4. Feature Engineering Strategy

The engineered features will be derived only from existing predictor variables.

The main categories considered are:

- URL length and character relationships
- Digit and special-character ratios
- Link density
- JavaScript and CSS relationships
- Domain and URL matching characteristics
- Other meaningful webpage structural relationships

Only features with a clear interpretation will be retained.

## 5. Leakage Prevention

The target variable `label` will not be used to create any engineered feature.

The metadata column `FILENAME` will also not be used.

All transformations will be based only on predictor information available before model training.

This ensures that the engineered feature set does not introduce information from the target variable and can be safely used in later machine learning stages.

## 6. Expected Outcome

At the end of this notebook, the project will contain an expanded feature set consisting of the original predictive features and carefully selected engineered features.

The resulting features will be checked for:

- Missing values
- Constant values
- Extreme or unusual distributions
- Correlation and redundancy
- Possible data leakage

The final engineered feature set will then be saved for use in the subsequent feature-selection and model-building stages.

In [1]:
# Importing libraries and loading the dataset

import pandas as pd
import numpy as np
import os

dataset_path = "../dataset/PhiUSIIL_Phishing_URL_Dataset.csv"

df = pd.read_csv(dataset_path)

print("FEATURE ENGINEERING ENVIRONMENT READY")
print("=" * 60)
print("Dataset shape:", df.shape)
print("Number of records:", len(df))
print("Number of columns:", len(df.columns))

FEATURE ENGINEERING ENVIRONMENT READY
Dataset shape: (235795, 56)
Number of records: 235795
Number of columns: 56


In [2]:
# Identify original predictive features

target_column = "label"
metadata_column = "FILENAME"

# Numerical predictive features only
original_features = [
    col for col in df.columns
    if col not in [target_column, metadata_column]
    and pd.api.types.is_numeric_dtype(df[col])
]

print("ORIGINAL PREDICTIVE FEATURES")
print("=" * 60)
print("Number of original numerical predictive features:", len(original_features))
print("\nFeature list:")
for i, feature in enumerate(original_features, start=1):
    print(f"{i:2d}. {feature}")

ORIGINAL PREDICTIVE FEATURES
Number of original numerical predictive features: 50

Feature list:
 1. URLLength
 2. DomainLength
 3. IsDomainIP
 4. URLSimilarityIndex
 5. CharContinuationRate
 6. TLDLegitimateProb
 7. URLCharProb
 8. TLDLength
 9. NoOfSubDomain
10. HasObfuscation
11. NoOfObfuscatedChar
12. ObfuscationRatio
13. NoOfLettersInURL
14. LetterRatioInURL
15. NoOfDegitsInURL
16. DegitRatioInURL
17. NoOfEqualsInURL
18. NoOfQMarkInURL
19. NoOfAmpersandInURL
20. NoOfOtherSpecialCharsInURL
21. SpacialCharRatioInURL
22. IsHTTPS
23. LineOfCode
24. LargestLineLength
25. HasTitle
26. DomainTitleMatchScore
27. URLTitleMatchScore
28. HasFavicon
29. Robots
30. IsResponsive
31. NoOfURLRedirect
32. NoOfSelfRedirect
33. HasDescription
34. NoOfPopup
35. NoOfiFrame
36. HasExternalFormSubmit
37. HasSocialNet
38. HasSubmitButton
39. HasHiddenFields
40. HasPasswordField
41. Bank
42. Pay
43. Crypto
44. HasCopyrightInfo
45. NoOfImage
46. NoOfCSS
47. NoOfJS
48. NoOfSelfRef
49. NoOfEmptyRef
50. NoOfE

## Feature Engineering Strategy

The original PhiUSIIL dataset already provides several useful URL and webpage characteristics. To improve the representation of phishing-related patterns, additional derived features will be created from the existing numerical predictors.

### 1. URL-Based Features

URL structure can provide useful information about suspicious websites. Derived features will represent:

- Character density within the URL.
- Ratio of digits to URL length.
- Ratio of special characters to URL length.
- Relationship between URL length and domain length.

### 2. Ratio and Density Features

Ratios normalize raw counts according to the size of the corresponding URL or webpage structure.

Examples include:

- Digits per URL character.
- Special characters per URL character.
- JavaScript-to-HTML relationship.
- CSS-to-HTML relationship.
- Iframe density.

### 3. Webpage Structural Features

Existing webpage characteristics can be combined to represent structural complexity, such as:

- JavaScript density.
- CSS density.
- Image density.
- Link density.
- Form-related density.

### 4. Feature Engineering Principles

The following rules will be followed:

1. Only existing predictor variables will be used.
2. The target variable `label` will never be used to create features.
3. `FILENAME` will not be used as a predictive feature.
4. Division-by-zero errors will be prevented.
5. Original features will be preserved.
6. Engineered features will be checked for missing and constant values.
7. Highly redundant features will be identified before final feature selection.

The purpose is not to create a large number of arbitrary variables, but to create a small set of interpretable features that capture relationships between existing measurements.

In [5]:
# Create URL-based engineered features

engineered_df = df.copy()

# Avoid division by zero
url_length = engineered_df["URLLength"].replace(0, np.nan)
domain_length = engineered_df["DomainLength"].replace(0, np.nan)

# 1. Digit density
engineered_df["DigitDensity"] = (
    engineered_df["NoOfDegitsInURL"] / url_length
)

# 2. Special-character density
engineered_df["SpecialCharDensity"] = (
    engineered_df["NoOfOtherSpecialCharsInURL"] / url_length
)

# 3. Equals-sign density
engineered_df["EqualsDensity"] = (
    engineered_df["NoOfEqualsInURL"] / url_length
)

# 4. Ampersand density
engineered_df["AmpersandDensity"] = (
    engineered_df["NoOfAmpersandInURL"] / url_length
)

# 5. URL-to-domain length ratio
engineered_df["URLDomainLengthRatio"] = (
    engineered_df["URLLength"] / domain_length
)

# Replace infinite values
engineered_df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("URL-BASED FEATURES CREATED")
print("=" * 60)

new_url_features = [
    "DigitDensity",
    "SpecialCharDensity",
    "EqualsDensity",
    "AmpersandDensity",
    "URLDomainLengthRatio"
]

for feature in new_url_features:
    print("-", feature)

print("\nOriginal columns:", df.shape[1])
print("Columns after engineering:", engineered_df.shape[1])

URL-BASED FEATURES CREATED
- DigitDensity
- SpecialCharDensity
- EqualsDensity
- AmpersandDensity
- URLDomainLengthRatio

Original columns: 56
Columns after engineering: 61


In [6]:
# Create ratio and density features

# Avoid division by zero
line_of_code = engineered_df["LineOfCode"].replace(0, np.nan)
url_length = engineered_df["URLLength"].replace(0, np.nan)

# 1. JavaScript density
engineered_df["JSDensity"] = (
    engineered_df["NoOfJS"] / line_of_code
)

# 2. CSS density
engineered_df["CSSDensity"] = (
    engineered_df["NoOfCSS"] / line_of_code
)

# 3. Image density
engineered_df["ImageDensity"] = (
    engineered_df["NoOfImage"] / line_of_code
)

# 4. Iframe density
engineered_df["IframeDensity"] = (
    engineered_df["NoOfiFrame"] / line_of_code
)

# 5. External reference density
engineered_df["ExternalRefDensity"] = (
    engineered_df["NoOfExternalRef"] / line_of_code
)

# 6. Self-reference density
engineered_df["SelfRefDensity"] = (
    engineered_df["NoOfSelfRef"] / line_of_code
)

# Replace infinite values
engineered_df.replace([np.inf, -np.inf], np.nan, inplace=True)

new_ratio_features = [
    "JSDensity",
    "CSSDensity",
    "ImageDensity",
    "IframeDensity",
    "ExternalRefDensity",
    "SelfRefDensity"
]

print("RATIO AND DENSITY FEATURES CREATED")
print("=" * 60)

for feature in new_ratio_features:
    print("-", feature)

print("\nTotal columns after engineering:", engineered_df.shape[1])

RATIO AND DENSITY FEATURES CREATED
- JSDensity
- CSSDensity
- ImageDensity
- IframeDensity
- ExternalRefDensity
- SelfRefDensity

Total columns after engineering: 67


In [7]:
# Create webpage structural features

# 1. Webpage feature presence score
# Counts important structural/security-related webpage indicators
engineered_df["WebStructureScore"] = (
    engineered_df["HasTitle"]
    + engineered_df["HasFavicon"]
    + engineered_df["IsResponsive"]
    + engineered_df["HasDescription"]
    + engineered_df["HasSubmitButton"]
    + engineered_df["HasPasswordField"]
    + engineered_df["HasHiddenFields"]
)

# 2. Redirect density relative to URL length
url_length = engineered_df["URLLength"].replace(0, np.nan)

engineered_df["RedirectDensity"] = (
    (engineered_df["NoOfURLRedirect"] +
     engineered_df["NoOfSelfRedirect"]) / url_length
)

# 3. Reference-to-code ratio
line_of_code = engineered_df["LineOfCode"].replace(0, np.nan)

engineered_df["TotalReferenceDensity"] = (
    (
        engineered_df["NoOfSelfRef"]
        + engineered_df["NoOfEmptyRef"]
        + engineered_df["NoOfExternalRef"]
    ) / line_of_code
)

# Replace infinite values
engineered_df.replace([np.inf, -np.inf], np.nan, inplace=True)

new_web_features = [
    "WebStructureScore",
    "RedirectDensity",
    "TotalReferenceDensity"
]

print("WEBPAGE STRUCTURAL FEATURES CREATED")
print("=" * 60)

for feature in new_web_features:
    print("-", feature)

print("\nTotal columns after engineering:", engineered_df.shape[1])

WEBPAGE STRUCTURAL FEATURES CREATED
- WebStructureScore
- RedirectDensity
- TotalReferenceDensity

Total columns after engineering: 70


In [8]:
# Verify newly created features

all_engineered_features = (
    new_url_features
    + new_ratio_features
    + new_web_features
)

print("ENGINEERED FEATURE VERIFICATION")
print("=" * 60)

print("Number of engineered features:", len(all_engineered_features))
print("Total dataset columns:", engineered_df.shape[1])

print("\nEngineered feature names:")
for i, feature in enumerate(all_engineered_features, start=1):
    print(f"{i:2d}. {feature}")

print("\nSample values:")
display(
    engineered_df[all_engineered_features].head()
)

ENGINEERED FEATURE VERIFICATION
Number of engineered features: 14
Total dataset columns: 70

Engineered feature names:
 1. DigitDensity
 2. SpecialCharDensity
 3. EqualsDensity
 4. AmpersandDensity
 5. URLDomainLengthRatio
 6. JSDensity
 7. CSSDensity
 8. ImageDensity
 9. IframeDensity
10. ExternalRefDensity
11. SelfRefDensity
12. WebStructureScore
13. RedirectDensity
14. TotalReferenceDensity

Sample values:


,DigitDensity,SpecialCharDensity,EqualsDensity,AmpersandDensity,URLDomainLengthRatio,JSDensity,CSSDensity,ImageDensity,IframeDensity,ExternalRefDensity,SelfRefDensity,WebStructureScore,RedirectDensity,TotalReferenceDensity
0,0.0,0.032258,0.0,0.0,1.291667,0.050179,0.035842,0.060932,0.001792,0.222222,0.213262,4,0.000000,0.435484
1,0.0,0.086957,0.0,0.0,1.437500,0.012945,0.014563,0.080906,0.000000,0.351133,0.063107,3,0.000000,0.414239
2,0.0,0.068966,0.0,0.0,1.318182,0.014989,0.004283,0.021413,0.000000,0.010707,0.089936,5,0.000000,0.104925
3,0.0,0.038462,0.0,0.0,1.368421,0.002360,0.004248,0.000472,0.001888,0.004877,0.003461,4,0.000000,0.008496
4,0.0,0.030303,0.0,0.0,1.269231,0.005584,0.002463,0.040072,0.000328,0.013960,0.011825,5,0.060606,0.025948


In [9]:
# Verify newly created features

all_engineered_features = (
    new_url_features
    + new_ratio_features
    + new_web_features
)

print("ENGINEERED FEATURE VERIFICATION")
print("=" * 60)

print("Number of engineered features:", len(all_engineered_features))
print("Total dataset columns:", engineered_df.shape[1])

print("\nEngineered feature names:")
for i, feature in enumerate(all_engineered_features, start=1):
    print(f"{i:2d}. {feature}")

print("\nSample values:")
display(
    engineered_df[all_engineered_features].head()
)

ENGINEERED FEATURE VERIFICATION
Number of engineered features: 14
Total dataset columns: 70

Engineered feature names:
 1. DigitDensity
 2. SpecialCharDensity
 3. EqualsDensity
 4. AmpersandDensity
 5. URLDomainLengthRatio
 6. JSDensity
 7. CSSDensity
 8. ImageDensity
 9. IframeDensity
10. ExternalRefDensity
11. SelfRefDensity
12. WebStructureScore
13. RedirectDensity
14. TotalReferenceDensity

Sample values:


,DigitDensity,SpecialCharDensity,EqualsDensity,AmpersandDensity,URLDomainLengthRatio,JSDensity,CSSDensity,ImageDensity,IframeDensity,ExternalRefDensity,SelfRefDensity,WebStructureScore,RedirectDensity,TotalReferenceDensity
0,0.0,0.032258,0.0,0.0,1.291667,0.050179,0.035842,0.060932,0.001792,0.222222,0.213262,4,0.000000,0.435484
1,0.0,0.086957,0.0,0.0,1.437500,0.012945,0.014563,0.080906,0.000000,0.351133,0.063107,3,0.000000,0.414239
2,0.0,0.068966,0.0,0.0,1.318182,0.014989,0.004283,0.021413,0.000000,0.010707,0.089936,5,0.000000,0.104925
3,0.0,0.038462,0.0,0.0,1.368421,0.002360,0.004248,0.000472,0.001888,0.004877,0.003461,4,0.000000,0.008496
4,0.0,0.030303,0.0,0.0,1.269231,0.005584,0.002463,0.040072,0.000328,0.013960,0.011825,5,0.060606,0.025948


In [10]:
# Check missing values and constant features in the engineered dataset

print("ENGINEERED FEATURE QUALITY CHECK")
print("=" * 60)

# 1. Missing values
missing_values = engineered_df.isnull().sum()

print("\nMissing values:")
print("Total missing values:", missing_values.sum())

if missing_values.sum() == 0:
    print("✅ No missing values found.")
else:
    print("\nColumns containing missing values:")
    display(missing_values[missing_values > 0])

# 2. Constant features
predictive_columns = [
    col for col in engineered_df.columns
    if col not in ["FILENAME", "URL", "Domain", "TLD", "Title", "label"]
]

constant_features = [
    col for col in predictive_columns
    if engineered_df[col].nunique(dropna=False) <= 1
]

print("\nConstant predictive features:", len(constant_features))

if len(constant_features) == 0:
    print("✅ No constant predictive features found.")
else:
    print("Constant features:")
    for feature in constant_features:
        print("-", feature)

# 3. Basic validity check for engineered features
print("\nEngineered feature validity:")
print("Finite values:", np.isfinite(
    engineered_df[all_engineered_features].select_dtypes(include=np.number)
).all().all())

print("\n✅ Feature quality check completed.")

ENGINEERED FEATURE QUALITY CHECK

Missing values:
Total missing values: 0
✅ No missing values found.

Constant predictive features: 0
✅ No constant predictive features found.

Engineered feature validity:
Finite values: True

✅ Feature quality check completed.


In [11]:
# Correlation and redundancy analysis

print("CORRELATION AND REDUNDANCY ANALYSIS")
print("=" * 60)

# Select numerical predictive features
correlation_columns = [
    col for col in engineered_df.columns
    if col not in ["FILENAME", "URL", "Domain", "TLD", "Title", "label"]
    and pd.api.types.is_numeric_dtype(engineered_df[col])
]

# Calculate correlation with target
target_correlation = (
    engineered_df[correlation_columns + ["label"]]
    .corr()["label"]
    .drop("label")
    .sort_values(key=abs, ascending=False)
)

print("\nTop 15 features correlated with the target:")
display(target_correlation.head(15).to_frame("Correlation"))

# Find highly correlated feature pairs
feature_corr = engineered_df[correlation_columns].corr().abs()

upper_triangle = feature_corr.where(
    np.triu(np.ones(feature_corr.shape), k=1).astype(bool)
)

high_correlation_pairs = []

for column in upper_triangle.columns:
    for row in upper_triangle.index:
        value = upper_triangle.loc[row, column]

        if pd.notna(value) and value >= 0.90:
            high_correlation_pairs.append(
                (row, column, value)
            )

print("\nHighly correlated feature pairs (|r| >= 0.90):")

if high_correlation_pairs:
    high_corr_df = pd.DataFrame(
        high_correlation_pairs,
        columns=["Feature 1", "Feature 2", "Absolute Correlation"]
    ).sort_values(
        "Absolute Correlation",
        ascending=False
    )

    display(high_corr_df)
else:
    print("No feature pairs with |r| >= 0.90 were found.")

print("\nTotal numerical predictive features analyzed:",
      len(correlation_columns))

CORRELATION AND REDUNDANCY ANALYSIS

Top 15 features correlated with the target:


,Correlation
URLSimilarityIndex,0.860358
HasSocialNet,0.784255
WebStructureScore,0.770542
HasCopyrightInfo,0.743358
HasDescription,0.690232
IsHTTPS,0.609132
DomainTitleMatchScore,0.584905
HasSubmitButton,0.578561
IsResponsive,0.548608
URLTitleMatchScore,0.539419



Highly correlated feature pairs (|r| >= 0.90):


,Feature 1,Feature 2,Absolute Correlation
2,DegitRatioInURL,DigitDensity,0.999998
3,SpacialCharRatioInURL,SpecialCharDensity,0.985712
1,DomainTitleMatchScore,URLTitleMatchScore,0.961008
0,URLLength,NoOfLettersInURL,0.956047
4,ExternalRefDensity,TotalReferenceDensity,0.904234
5,SelfRefDensity,TotalReferenceDensity,0.903719



Total numerical predictive features analyzed: 64


In [12]:
# Prepare final engineered feature list and perform leakage check

target_column = "label"
excluded_columns = [
    "FILENAME",
    "URL",
    "Domain",
    "TLD",
    "Title",
    target_column
]

# Final predictive features
final_engineered_features = [
    col for col in engineered_df.columns
    if col not in excluded_columns
    and pd.api.types.is_numeric_dtype(engineered_df[col])
]

print("FINAL FEATURE LIST AND LEAKAGE CHECK")
print("=" * 60)

print("Total predictive features:", len(final_engineered_features))

print("\nExcluded columns:")
for col in excluded_columns:
    print("-", col)

# Leakage check
leakage_columns = [
    col for col in final_engineered_features
    if col == target_column
]

print("\nTarget leakage check:")
if len(leakage_columns) == 0:
    print("✅ No target variable leakage detected.")
else:
    print("❌ Target variable found in predictive features:", leakage_columns)

# Verify metadata is excluded
metadata_check = [
    col for col in final_engineered_features
    if col == "FILENAME"
]

print("\nMetadata leakage check:")
if len(metadata_check) == 0:
    print("✅ FILENAME is excluded from predictive features.")
else:
    print("❌ FILENAME found in predictive features.")

print("\nFinal feature names:")
for i, feature in enumerate(final_engineered_features, start=1):
    print(f"{i:2d}. {feature}")

FINAL FEATURE LIST AND LEAKAGE CHECK
Total predictive features: 64

Excluded columns:
- FILENAME
- URL
- Domain
- TLD
- Title
- label

Target leakage check:
✅ No target variable leakage detected.

Metadata leakage check:
✅ FILENAME is excluded from predictive features.

Final feature names:
 1. URLLength
 2. DomainLength
 3. IsDomainIP
 4. URLSimilarityIndex
 5. CharContinuationRate
 6. TLDLegitimateProb
 7. URLCharProb
 8. TLDLength
 9. NoOfSubDomain
10. HasObfuscation
11. NoOfObfuscatedChar
12. ObfuscationRatio
13. NoOfLettersInURL
14. LetterRatioInURL
15. NoOfDegitsInURL
16. DegitRatioInURL
17. NoOfEqualsInURL
18. NoOfQMarkInURL
19. NoOfAmpersandInURL
20. NoOfOtherSpecialCharsInURL
21. SpacialCharRatioInURL
22. IsHTTPS
23. LineOfCode
24. LargestLineLength
25. HasTitle
26. DomainTitleMatchScore
27. URLTitleMatchScore
28. HasFavicon
29. Robots
30. IsResponsive
31. NoOfURLRedirect
32. NoOfSelfRedirect
33. HasDescription
34. NoOfPopup
35. NoOfiFrame
36. HasExternalFormSubmit
37. HasSoci

In [ ]:
# Final verification and save engineered dataset

print("FINAL FEATURE ENGINEERING VERIFICATION")
print("=" * 60)

# Dataset information
print("Final dataset shape:", engineered_df.shape)
print("Total records:", len(engineered_df))
print("Total columns:", len(engineered_df.columns))

# Predictive feature information
print("\nOriginal predictive features:", len(original_features))
print("Engineered features:", len(all_engineered_features))
print("Final predictive features:", len(final_engineered_features))

# Data quality checks
print("\nData quality:")
print("Missing values:", engineered_df.isnull().sum().sum())
print("Duplicate rows:", engineered_df.duplicated().sum())

# Target distribution
print("\nTarget distribution:")
print(engineered_df["label"].value_counts().sort_index())

# Verify excluded columns
print("\nExcluded non-predictive columns:")
print(excluded_columns)

# Save engineered dataset
output_path = "../dataset/PhiUSIIL_Engineered_Features.csv"

engineered_df.to_csv(output_path, index=False)

print("\n" + "=" * 60)
print("✅ FEATURE ENGINEERING COMPLETED")
print("Engineered dataset saved to:")
print(output_path)